# CHECK_BLOWER_TREES

* Is the "without blower" result limited by early stopping?
* In that run the "Without blower" fits stopped after 1 to 10 trees (a single tree at W60), so their scores may understate what the channels can do. Here the same XGBoost is fitted on the full training partition with a **fixed** number of trees (100 and 300) and no early stopping, for the primary target and for the target without blower onsets. The test partition is scored once per fit; neither tree count is selected on it, both are reported.
* Read-only; 16 fits. 
* Output:
    * `blower_trees_check.csv`

In [ ]:
import numpy as np, pandas as pd, warnings
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

CFG = dict(
    labelled   = "TELEMETRY/build/telemetry_labelled_v2.pkl",
    episodes   = "TELEMETRY/build/fault_episodes_v2.csv",
    outdir     = "OUTPUT",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons   = ["D5", "W5", "W60", "W300"],
    windows    = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    patience   = 100, test_frac = 0.20, val_frac = 0.20, seed = 42,
    sample_frac = 1.0
)
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
TRACTION_MODES = ["POWERING", "PROPULSION", "REGENERATIVE BRAKING", "VCB ON"]

tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
if CFG["sample_frac"] < 1.0:
    tel = tel.iloc[:int(len(tel) * CFG["sample_frac"])].copy()
t0, t1 = tel["DATETIME"].min(), tel["DATETIME"].max()
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep = ep[(ep.onset >= t0) & (ep.onset <= t1)].copy()
ep["is_blower"] = ep["fault"].eq(CFG["blower"])
print(f"{len(tel):,} telemetry rows, {t0} to {t1}")
print(f"{len(ep)} onsets in that period, {ep.is_blower.sum()} blower ({100*ep.is_blower.mean():.1f}%)")

In [ ]:
ts = tel["DATETIME"].values.astype("datetime64[s]")
pos = pd.Series(np.arange(len(tel)), index=pd.DatetimeIndex(ts))

def label_before(onsets, h, lag=1):
    """Label of horizon h at the second `lag` s before each onset (NaN if no telemetry)."""
    idx = pos.reindex(pd.DatetimeIndex(onsets) - pd.Timedelta(seconds=lag))
    return tel[h].values[idx.dropna().astype(int)], idx.notna().values

rows = []
for keep in [True, False]:
    sub = ep[ep.debounce_keep == keep]
    lab, has = label_before(sub.onset, "W5")
    rows.append(dict(debounce_keep=keep, onsets=len(sub), with_telemetry=int(has.sum()), W5_positive_pct=round(100 * np.mean(lab == 1), 1),W5_excluded_pct=round(100 * np.mean(lab == -1), 1)))
deb = pd.DataFrame(rows).set_index("debounce_keep")
print(deb.to_string())
USE_ALL = deb.loc[False, "W5_positive_pct"] > 0.5 * deb.loc[True, "W5_positive_pct"]
used = ep if USE_ALL else ep[ep.debounce_keep]
print(f"\n-> LABELS2 appears to use {'ALL onsets' if USE_ALL else 'only debounce_keep = True onsets'}; "
      f"{len(used)} onsets used below ({used.is_blower.sum()} blower).")
deb.to_csv(f"{CFG['outdir']}/debounce_check.csv")

In [ ]:
def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y):
    n = len(y); t = int(n * (1 - CFG["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], CFG["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def xgb(y, n_iter=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
    return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                         random_state=CFG["seed"], n_jobs=-1, **kw)

def fit_score(X, y):
    ia, iv, itr, ite = three_way_split(y)
    m1 = xgb(y[ia]).fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
    n_iter = int(m1.best_iteration) + 1
    m2 = xgb(y[itr], n_iter).fit(X[itr], y[itr])
    s = m2.predict_proba(X[ite])[:, 1]
    base = y[ite].mean()
    ap = average_precision_score(y[ite], s)
    return dict(pr_auc=round(ap, 3), baseline=round(base, 4), lift=round(ap / base, 1), test_positive_s=int(y[ite].sum()), trees=n_iter)

def coverage(onsets, lo, hi):
    """Number of onset windows [onset+lo, onset+hi] covering each telemetry row."""
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]


def fit_fixed(X, y, n_trees):
    ia, iv, itr, ite = three_way_split(y)
    m = xgb(y[itr], n_trees).fit(X[itr], y[itr])
    s = m.predict_proba(X[ite])[:, 1]
    base = y[ite].mean(); ap = average_precision_score(y[ite], s)
    return dict(pr_auc=round(ap, 3), baseline=round(base, 4), lift=round(ap / base, 1), test_positive_s=int(y[ite].sum()))

X_all = tel[PRIMARY].astype("float32").fillna(0).values
res = []
for h in tqdm(CFG["horizons"], desc="horizons"):
    lo, hi = CFG["windows"][h]
    cov_b = coverage(used.onset[used.is_blower], lo, hi)
    cov_o = coverage(used.onset[~used.is_blower], lo, hi)
    y0 = tel[h].astype(int).values
    variants = {"All onsets (primary)": y0,
                "Without blower": np.where((y0 == 1) & (cov_b > 0) & (cov_o == 0), -1, y0)}
    for name, y in variants.items():
        keep = y != -1
        for n_trees in [100, 300]:
            r = fit_fixed(X_all[keep], y[keep], n_trees)
            r.update(horizon=h, variant=name, trees=n_trees); res.append(r)
            tqdm.write(f"  {h:5s} {name:22s} {n_trees:4d} trees  PR-AUC {r['pr_auc']:.3f}  "
                       f"baseline {r['baseline']:.4f}  lift {r['lift']:.1f}x")
res = pd.DataFrame(res)
out = res.pivot_table(index=["variant", "trees"], columns="horizon", values="pr_auc")[CFG["horizons"]]
print("\nPR-AUC\n", out.to_string())
res.to_csv(f"{CFG['outdir']}/blower_trees_check.csv", index=False)

___
**END**